## CSC311 Project: Model Training

This notebook trains a Multinomial Logistic Regression model on the project dataset. It performs the following steps:
1.  Loads the data and identifies feature types.
2.  Splits the data into training and validation sets, ensuring responses from the same user stay in the same set.
3.  Builds a preprocessing pipeline using `scikit-learn` that:
    *   Imputes missing values.
    *   Applies TF-IDF to text features.
    *   One-hot encodes categorical features.
    *   Standardizes the final feature matrix.
4.  Trains a `LogisticRegression` model.
5.  Evaluates the model on the validation set.
6.  Exports all necessary assets (model weights, preprocessing stats, etc.) to JSON and NumPy files for use in `pred.py`.

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

### 1. Load Data and Define Features

In [2]:
# Load data
df = pd.read_csv('training_data_202601.csv')

# Rename columns for easier access
df.rename(columns={
    'Painting': 'artist',
    'unique_id': 'user_id',
    'On a scale of 1–10, how intense is the emotion conveyed by the artwork?': 'impression_rating',
    'How much (in Canadian dollars) would you be willing to pay for this painting?': 'price',
    'What season does this art piece remind you of?': 'season',
    'If you could purchase this painting, which room would you put that painting in?': 'room',
    'Describe how this painting makes you feel.': 'moods_text',
    'Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.': 'story_text'
}, inplace=True)

# Force numerical columns to numeric type, coercing errors to NaN
for col in ['impression_rating', 'price']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Define feature types based on the new column names
TARGET = 'artist'

# Note: I'm selecting a subset of numerical features that seem most relevant.
# The original 'like_rating', 'interest_rating', 'overall_rating' are not in this CSV.
NUMERICAL_FEATURES = [
    'impression_rating',
    'price'
]

# Note: 'time_of_day' is not in this CSV.
CATEGORICAL_FEATURES = [
    'season',
    'room'
]

TEXT_FEATURES = [
    'moods_text',
    'story_text'
]

# For simplicity, we'll combine the text features into one
df['combined_text'] = df[TEXT_FEATURES].fillna('').agg(' '.join, axis=1)
SINGLE_TEXT_FEATURE = 'combined_text'

### 2. Grouped Train-Validation Split

We split by `user_id` to prevent data leakage.

In [3]:
user_ids = df['user_id'].unique()
train_user_ids, val_user_ids = train_test_split(user_ids, test_size=0.2, random_state=42)

train_df = df[df['user_id'].isin(train_user_ids)]
val_df = df[df['user_id'].isin(val_user_ids)]

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_val = val_df.drop(columns=[TARGET])
y_val = val_df[TARGET]

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")

Training set size: 1347
Validation set size: 339


### 3. Build Preprocessing and Model Pipeline

In [4]:
# Create preprocessing pipelines for each feature type
numeric_transformer = SimpleImputer(strategy='mean')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
text_transformer = TfidfVectorizer(max_features=500, stop_words='english')

# Create a preprocessor object using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERICAL_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES),
        ('text', text_transformer, SINGLE_TEXT_FEATURE)
    ],
    remainder='drop'
)

# Create the full model pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler(with_mean=False)), # Use with_mean=False for sparse matrices from TF-IDF
    ('classifier', LogisticRegression(solver='lbfgs', C=1.0, random_state=42, max_iter=1000))
])

### 4. Train and Evaluate the Model 1

In [5]:
model_pipeline.fit(X_train, y_train)

# Evaluate on validation set
y_pred = model_pipeline.predict(X_val)
print(classification_report(y_val, y_pred))

                           precision    recall  f1-score   support

The Persistence of Memory       0.84      0.81      0.83       113
         The Starry Night       0.73      0.72      0.72       113
      The Water Lily Pond       0.79      0.82      0.81       113

                 accuracy                           0.78       339
                macro avg       0.78      0.78      0.78       339
             weighted avg       0.78      0.78      0.78       339



### 5. Train and Evaluate the Model 2

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# --- Preprocess the data first ---
# We need to fit the preprocessor on the training data and transform both train and val sets
print("Fitting preprocessor and transforming data...")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# Also, encode the labels to be sure they are numeric for all models
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)


# --- 2. Random Forest Classifier ---
print("\n--- Starting Random Forest Tuning ---")

# Define the parameter grid based on your proposal
param_grid_rf = {
    'n_estimators': [100, 300],       # Reduced for faster tuning, you can add 500 back
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 5]
}

# Initialize the Random Forest classifier
# Note: We don't need the scaler for Random Forest
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Set up GridSearchCV
# Using 3-fold cross-validation on the processed training data
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=3, verbose=2, scoring='f1_macro')

# Fit GridSearchCV to the processed training data
print("Starting GridSearchCV for Random Forest...")
grid_search_rf.fit(X_train_processed, y_train_encoded)

# Get the best estimator
best_rf = grid_search_rf.best_estimator_

# Print the best parameters found
print("\nBest Random Forest Parameters:")
print(grid_search_rf.best_params_)

# Evaluate the best model on the processed validation set
print("\nEvaluating Random Forest on the validation set...")
y_val_pred_rf = best_rf.predict(X_val_processed)

# Calculate and print metrics
accuracy_rf = accuracy_score(y_val_encoded, y_val_pred_rf)
f1_rf = f1_score(y_val_encoded, y_val_pred_rf, average='macro')

print(f"Validation Accuracy (Random Forest): {accuracy_rf:.4f}")
print(f"Validation Macro F1-Score (Random Forest): {f1_rf:.4f}")

# Print a classification report
print("\nClassification Report (Random Forest):")
print(classification_report(y_val_encoded, y_val_pred_rf, target_names=label_encoder.classes_))

Fitting preprocessor and transforming data...

--- Starting Random Forest Tuning ---
Starting GridSearchCV for Random Forest...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   0.4s
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   0.4s
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   0.4s
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=300; total time=   1.1s
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=300; total time=   1.1s
[CV] END max_depth=None, min_samples_leaf=1, n_estimators=300; total time=   1.0s
[CV] END max_depth=None, min_samples_leaf=5, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, min_samples_leaf=5, n_estimators=100; total time=   0.2s
[CV] END max_depth=None, min_samples_leaf=5, n_estimators=100; total time=   0.3s
[CV] END max_depth=None, min_samples_leaf=5, n_estimators=300; total time

### 6. Train and Evaluate the Model 3

In [7]:
# --- 3. Feedforward Neural Network (MLP) ---
from sklearn.neural_network import MLPClassifier

print("\n--- Starting Neural Network (MLP) Tuning ---")

# Define the parameter grid based on your proposal
# We'll test one and two hidden layers
param_grid_mlp = {
    'hidden_layer_sizes': [(64,), (128,), (64, 64)],
    'learning_rate_init': [0.001, 0.01],
    'alpha': [0.0001, 0.001] # L2 regularization (weight decay)
}

# Initialize the MLPClassifier
# We set a higher max_iter and use early_stopping as described in the proposal
mlp = MLPClassifier(random_state=42,
                    max_iter=100,
                    early_stopping=True,
                    validation_fraction=0.15, # Use 15% of training data for early stopping
                    n_iter_no_change=10,      # Patience of 10 epochs
                    solver='adam')

# Set up GridSearchCV
# This will be slower than the other models
grid_search_mlp = GridSearchCV(estimator=mlp, param_grid=param_grid_mlp, cv=3, verbose=2, scoring='f1_macro')

# Fit GridSearchCV to the processed training data
print("Starting GridSearchCV for MLP...")
grid_search_mlp.fit(X_train_processed, y_train_encoded)

# Get the best estimator
best_mlp = grid_search_mlp.best_estimator_

# Print the best parameters found
print("\nBest MLP Parameters:")
print(grid_search_mlp.best_params_)

# Evaluate the best model on the processed validation set
print("\nEvaluating MLP on the validation set...")
y_val_pred_mlp = best_mlp.predict(X_val_processed)

# Calculate and print metrics
accuracy_mlp = accuracy_score(y_val_encoded, y_val_pred_mlp)
f1_mlp = f1_score(y_val_encoded, y_val_pred_mlp, average='macro')

print(f"Validation Accuracy (MLP): {accuracy_mlp:.4f}")
print(f"Validation Macro F1-Score (MLP): {f1_mlp:.4f}")

# Print a classification report
print("\nClassification Report (MLP):")
print(classification_report(y_val_encoded, y_val_pred_mlp, target_names=label_encoder.classes_))


--- Starting Neural Network (MLP) Tuning ---
Starting GridSearchCV for MLP...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   0.6s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   0.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   0.2s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=   0.2s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=   0.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=   0.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   0.7s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   0.4s
[CV] END alpha=0.0001, hidden_layer_sizes=(128,), learning_rate_init=0.001; total time=   0.4s
[CV] END alpha

### 7. Export Model with Best Performance


In [ ]:
# --- Final Step: Export Best Model (Random Forest) and Preprocessor ---
import joblib
import pandas as pd

# 1. Identify the best model's parameters from the grid search
best_rf_params = grid_search_rf.best_params_
print(f"Best Random Forest parameters: {best_rf_params}")

# 2. Re-train the preprocessor and the best model on the COMBINED training and validation data
print("\nRe-training the final preprocessor and Random Forest model on all available training data (train + val)...")

# Combine the training and validation sets
X_train_full = pd.concat([X_train, X_val], ignore_index=True)
y_train_full = pd.concat([y_train, y_val], ignore_index=True)

# Fit the preprocessor on the full training data
final_preprocessor = preprocessor.fit(X_train_full)

# Transform the full training data
X_train_full_processed = final_preprocessor.transform(X_train_full)

# Encode the labels for the full training data
final_label_encoder = LabelEncoder().fit(y_train_full)
y_train_full_encoded = final_label_encoder.transform(y_train_full)

# Create and train the final Random Forest model
final_rf_model = RandomForestClassifier(**best_rf_params, random_state=42, n_jobs=-1)
final_rf_model.fit(X_train_full_processed, y_train_full_encoded)

print("Final model training complete.")

# 3. Save the final assets using joblib
joblib.dump(final_rf_model, 'final_model.joblib')
joblib.dump(final_preprocessor, 'final_preprocessor.joblib')
joblib.dump(final_label_encoder, 'final_label_encoder.joblib')

print("\n✅ Successfully exported:")
print("- final_model.joblib")
print("- final_preprocessor.joblib")
print("- final_label_encoder.joblib")